# SleepFM LOO-CL 코드 구조 워크스루

이 노트북은 실행 결과를 요약하는 대신, **이번 분석이 코드상 어디서 시작해서 어디로 흘러가는지**를 따라가기 위한 노트북이다.

읽는 순서는 아래처럼 잡았다.

1. 관련 파일이 repo 안에서 어디에 있는지 보기
2. 모델이 `5분 pooled embedding`과 `5초 sequence embedding`을 어떻게 만드는지 보기
3. pretraining에서 LOO-CL loss가 실제로 어떻게 계산되는지 보기
4. embedding이 HDF5로 어떻게 저장되는지 보기
5. downstream diagnosis dataset과 evaluation 출력이 어떻게 만들어지는지 보기
6. geometry audit 스크립트가 어떤 계산을 하는지 보기
7. diagnosis join / figure 생성 스크립트가 어떤 구조인지 보기

즉, 이 노트북의 목적은 `결과를 보는 것`보다 `코드를 이해하는 것`에 있다.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SLEEPFM_CLINICAL = ROOT.parent / 'sleepfm-clinical'

FILES = {
    'models': SLEEPFM_CLINICAL / 'sleepfm' / 'models' / 'models.py',
    'dataset': SLEEPFM_CLINICAL / 'sleepfm' / 'models' / 'dataset.py',
    'pretrain': SLEEPFM_CLINICAL / 'sleepfm' / 'pipeline' / 'pretrain.py',
    'generate_embeddings': SLEEPFM_CLINICAL / 'sleepfm' / 'pipeline' / 'generate_embeddings.py',
    'evaluate_disease_prediction': SLEEPFM_CLINICAL / 'sleepfm' / 'pipeline' / 'evaluate_disease_prediction.py',
    'analyze_loo_geometry': ROOT / 'scripts' / 'analyze_loo_geometry.py',
    'make_loo_diagnosis_figures': ROOT / 'scripts' / 'make_loo_diagnosis_figures.py',
}

for key, path in FILES.items():
    print(f'{key:>28}: {path} | exists={path.exists()}')

def show_file_lines(path: Path, start: int, end: int):
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    text = '\n'.join(f'{idx:>4}: {lines[idx - 1]}' for idx in range(start, end + 1))
    print(text)

## 1. 관련 파일 지도

먼저 이번 분석에 직접 연결된 파일만 추려 보면 구조는 대략 아래와 같다.

- `sleepfm-clinical/sleepfm/models/models.py`
  - `SetTransformer`와 downstream model class 정의
- `sleepfm-clinical/sleepfm/pipeline/pretrain.py`
  - LOO-CL / pairwise contrastive objective 정의
- `sleepfm-clinical/sleepfm/pipeline/generate_embeddings.py`
  - pretrained model로 `5분`, `5초` embedding HDF5 생성
- `sleepfm-clinical/sleepfm/models/dataset.py`
  - downstream diagnosis dataset / collate 구조 정의
- `sleepfm-clinical/sleepfm/pipeline/evaluate_disease_prediction.py`
  - test split에서 `all_outputs`, `all_paths` 등 저장
- `sleepfm-repro/scripts/analyze_loo_geometry.py`
  - `5분 aggregated embedding`으로 geometry metric 계산
- `sleepfm-repro/scripts/make_loo_diagnosis_figures.py`
  - diagnosis 결과와 geometry를 join하고 figure 생성

이걸 dataflow로 쓰면 아래 한 줄이다.

`SetTransformer -> generate_embeddings -> shhs_5min_agg/*.hdf5 -> analyze_loo_geometry -> subject/window metrics -> evaluate_disease_prediction outputs -> make_loo_diagnosis_figures -> join.csv + png`

## 2. 모델 구조: `SetTransformer.forward()`가 실제로 뭘 반환하는가

LOO-CL 분석을 하려면 가장 먼저 `어떤 latent를 보고 있는가`가 분명해야 한다.

핵심은 `SetTransformer.forward()`가 두 가지를 반환한다는 점이다.

- `x`: temporal pooling 이후의 5분 pooled embedding
- `embedding`: temporal pooling 직전의 5초 sequence embedding

In [ ]:
show_file_lines(FILES['models'], 105, 152)

### 여기서 봐야 할 것

- `patch_embedding`으로 signal patch를 token화한다.
- `spatial_pooling`으로 modality 내부 채널 축을 먼저 줄인다.
- `transformer_encoder`가 시간축 representation을 만든다.
- `embedding = x.clone()`가 저장되는 시점이 `5초 sequence latent`다.
- 그 다음 `temporal_pooling`을 거친 `x`가 `5분 pooled latent`다.

즉, 현재 LOO-CL geometry audit는 `return x, embedding` 중 첫 번째 값인 `x`를 보고 있는 것이다.

## 3. pretraining 구조: LOO-CL loss가 어디서 계산되는가

이제 contrastive objective가 실제로 어떻게 구현되어 있는지 본다. 핵심 함수는 `run_iter()`다.

In [ ]:
show_file_lines(FILES['pretrain'], 15, 99)

### 여기서 봐야 할 것

- modality별 input을 각각 `model(...)`에 통과시켜 pooled embedding을 만든다.
- `emb = [e[0] for e in emb]`로 pooled embedding만 뽑는다.
- 각 embedding은 **평균 전에** `torch.nn.functional.normalize`로 L2 normalize 된다.
- `mode == 'leave_one_out'`일 때 `other_emb`를 나머지 modality 평균으로 만든다.
- query-to-other, other-to-query 양방향 cross-entropy를 모두 더한다.

즉, 논문 수식 수준에서 궁금했던 `평균 전에 normalize 하느냐`는 질문에 대해서, 현재 공개 구현은 **그렇다**가 답이다.

## 4. embedding 저장 구조: `generate_embeddings.py`

다음으로 봐야 할 것은, 이 latent가 실제 파일로 어떻게 저장되는가이다. 여기서 `5분 aggregated`와 `5초 sequence`가 나뉜다.

In [ ]:
show_file_lines(FILES['generate_embeddings'], 120, 175)

### 저장 구조 해석

- `embeddings_new = [e[0].unsqueeze(1) for e in embeddings]`는 pooled embedding을 `*_5min_agg` 경로에 저장한다.
- `embeddings_new = [e[1] for e in embeddings]`는 sequence embedding을 일반 embedding 경로에 저장한다.
- 각 subject별로 HDF5 하나가 생기고, 그 안에 modality dataset (`BAS`, `RESP`, `EKG`, `EMG`)가 들어간다.

그래서 우리가 geometry audit할 때 입력으로 쓰는 디렉터리는 `artifacts/embeddings/.../shhs_5min_agg`다.

## 5. downstream diagnosis 입력 구조: dataset / collate

geometry와 diagnosis를 연결하려면, downstream evaluation이 어떤 subject 단위로 돌아가는지 알아야 한다.

핵심은 `DiagnosisFinetuneFullCOXPHWithDemoDataset`과 그 collate 함수다.

In [ ]:
show_file_lines(FILES['dataset'], 516, 651)

### diagnosis dataset 구조 해석

- split JSON에서 train / validation / test용 HDF5 path를 읽는다.
- `is_event.csv`, `time_to_event.csv`, `demo_labels.csv`를 `Study ID` 기준으로 join한다.
- 실제 input은 subject별 embedding HDF5이며, modality dataset들을 읽어 `x_data`로 만든다.
- collate 단계에서 modality 수와 sequence 길이를 패딩해서 batch로 만든다.
- 최종적으로 model에 들어가는 것은 `x_data`, `event_time`, `is_event`, `demo_feats`, `padded_mask`, `hdf5_path_list`다.

즉, downstream evaluation은 **subject 단위**로 돌아가고, geometry join도 자연스럽게 subject 단위가 된다.

## 6. downstream evaluation 출력 구조: `evaluate_disease_prediction.py`

다음은 test split에서 어떤 아티팩트가 저장되는지 본다.

In [ ]:
show_file_lines(FILES['evaluate_disease_prediction'], 110, 166)

### evaluation 출력 해석

이 함수는 test split을 한 바퀴 돌면서 아래 파일을 저장한다.

- `all_outputs.pickle`: subject x target hazard output
- `all_event_times.pickle`: subject x target event time
- `all_is_event.pickle`: subject x target censor/event indicator
- `all_paths.pickle`: 각 subject에 대응하는 embedding HDF5 path

바로 이 `all_paths.pickle`이 geometry table과 downstream prediction을 묶는 join key 역할을 한다.

## 7. geometry audit 스크립트 구조: `analyze_loo_geometry.py`

이제 reproduction repo에 추가한 geometry audit 스크립트를 본다. 이 파일의 목적은 `5분 pooled embedding HDF5 -> subject/window metric CSV` 변환이다.

In [ ]:
show_file_lines(FILES['analyze_loo_geometry'], 1, 122)

In [ ]:
show_file_lines(FILES['analyze_loo_geometry'], 123, 251)

### geometry script 구조 해석

핵심 함수는 세 개다.

- `load_subject_embeddings()`
  - subject HDF5에서 modality별 `[n_windows, embed_dim]` 배열을 읽는다.
- `compute_metrics()`
  - modality를 normalize한 뒤 `consensus_norm`, `loo_mean_norm_mean`, `alignment_mean`, `jackknife_instability`를 계산한다.
- `main()`
  - 모든 subject를 순회하면서 `window_metrics.csv`, `subject_metrics.csv`, `summary.json`을 쓴다.

즉, 이 스크립트는 modeling code를 건드리지 않고, 이미 생성된 latent만 읽어서 geometry를 계량화하는 후처리 단계다.

## 8. diagnosis join / figure 생성 구조: `make_loo_diagnosis_figures.py`

다음 스크립트는 geometry 결과와 diagnosis 결과를 합치고, figure를 만든다.

In [ ]:
show_file_lines(FILES['make_loo_diagnosis_figures'], 1, 190)

In [ ]:
show_file_lines(FILES['make_loo_diagnosis_figures'], 191, 331)

### figure script 구조 해석

이 파일은 크게 네 단계로 나뉜다.

1. evaluation pickles와 geometry CSV를 읽는다.
2. `6년 horizon event`를 만들고, target별 `risk`, `error_type(TP/TN/FP/FN)`를 계산한다.
3. subject-level geometry와 downstream 결과를 join해서 `diagnosis_geometry_join.csv`를 만든다.
4. scatter / boxplot / risk-vs-instability / constellation figure를 그린다.

즉, 이 스크립트는 `분석 표준화 + 그림 생성` 역할을 맡는다.

## 9. 코드 레벨 dataflow를 한 번에 정리하면

아래 셀은 지금까지 본 흐름을 아주 짧게 다시 적은 것이다.

In [ ]:
flow = [
    ('models.py', 'SetTransformer.forward() returns (pooled_5min, sequence_5sec)'),
    ('pretrain.py', 'run_iter() normalizes pooled embeddings and computes leave-one-out contrastive loss'),
    ('generate_embeddings.py', 'writes pooled embeddings to shhs_5min_agg/*.hdf5'),
    ('dataset.py', 'diagnosis dataset reads subject-level embedding HDF5 + labels + demo features'),
    ('evaluate_disease_prediction.py', 'writes all_outputs / all_event_times / all_is_event / all_paths'),
    ('analyze_loo_geometry.py', 'reads shhs_5min_agg HDF5 and writes geometry CSV/JSON'),
    ('make_loo_diagnosis_figures.py', 'joins geometry + diagnosis outputs and writes final tables/figures'),
]

for step, desc in flow:
    print(f'- {step}: {desc}')

## 10. 실제 artifact 경로까지 연결해서 보기

코드 구조가 실제 산출물과 어떻게 맞물리는지 보려면 아래 경로들을 같이 보면 된다.

In [ ]:
artifact_paths = {
    '5min embeddings': ROOT / 'artifacts' / 'embeddings' / 'model_base_paperexact' / 'shhs_5min_agg',
    'geometry summary': ROOT / 'artifacts' / 'audit' / 'loo_geometry_model_base_paperexact_shhs' / 'summary.json',
    'geometry subject csv': ROOT / 'artifacts' / 'audit' / 'loo_geometry_model_base_paperexact_shhs' / 'subject_metrics.csv',
    'diagnosis outputs': ROOT / 'artifacts' / 'embeddings' / 'model_base_paperexact' / 'DiagnosisFinetuneFullLSTMCOXPHWithDemo_shhs_shhs_6cvd_paperexact_3291_496_2000_demo_labels_BAS_RESP_EKG_EMG__ep_5_bs_8' / 'shhs_downstream_paperlike_3291_496_2000' / 'test',
    'diagnosis-geometry join': ROOT / 'artifacts' / 'audit' / 'loo_geometry_model_base_paperexact_shhs' / 'diagnosis_6y' / 'diagnosis_geometry_join.csv',
    'example figure': ROOT / 'artifacts' / 'audit' / 'loo_geometry_model_base_paperexact_shhs' / 'diagnosis_6y' / 'chf_example_constellations.png',
}

for name, path in artifact_paths.items():
    print(f'{name:>24}: {path} | exists={path.exists()}')

## 11. 이 노트북을 어떻게 읽으면 좋은가

지금 단계에서 가장 좋은 읽는 방법은 아래 순서다.

1. `models.py`와 `pretrain.py`를 먼저 보고, 어떤 latent에 LOO-CL이 걸리는지 이해한다.
2. `generate_embeddings.py`를 보고 그 latent가 어떤 HDF5로 저장되는지 이해한다.
3. `dataset.py`와 `evaluate_disease_prediction.py`를 보고 downstream 결과가 subject 단위로 저장되는 걸 확인한다.
4. `analyze_loo_geometry.py`를 보고 geometry metric 계산 공식을 코드에서 확인한다.
5. `make_loo_diagnosis_figures.py`를 보고 geometry와 diagnosis가 어떻게 합쳐지는지 이해한다.

즉, 이 노트북은 `왜 이런 결과가 나왔는지`를 코드 단위로 따라갈 때 쓰면 된다.

## 12. 다음 단계 제안

코드 구조까지 이해한 다음 가장 자연스러운 다음 단계는 둘 중 하나다.

- `window-level metadata(stage transition, respiratory event density, arousal density)`를 붙이는 새 스크립트 추가
- `constellation plot` 사례를 자동으로 더 많이 뽑고, 예시 case table까지 만드는 확장 노트북 추가

원하면 다음 턴에서 그 구조까지 이어서 만들 수 있다.